Tema 3 - Multi Layer Perceptron

Incarcam datele

In [ ]:
import numpy as np

# train = np.load("/kaggle/input/competitions/nn-2026-fii-homework-3/train.npz")
# test = np.load("/kaggle/input/competitions/nn-2026-fii-homework-3/test.npz")

train = np.load("./nn-2026-fii-homework-3/train.npz")
test = np.load("./nn-2026-fii-homework-3/test.npz")

X_train = train["data"] / 255.0
y_train = train["labels"]
X_test = test["data"] / 255.0

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

# He Initialziaiton
w1 = np.random.randn(784, 100) * np.sqrt(2.0 / 784)  # (784,100)
b1 = np.zeros((1,100))                               # (1,100)

w2 = np.random.randn(100, 10) * np.sqrt(2.0 / 100)   # (100,10)
b2 = np.zeros((1, 10))                               # (1,10)

n = X_train.shape[0]


(60000, 784)
(60000,)
(15000, 784)


Functia ReLU si derivata ei, si softmax

In [2]:
def ReLU(z:np.array)->np.array:
    return np.maximum(0,z)

def deriv_ReLU(y:np.array)->np.array:
    return (y>0).astype(float)

def softmax(z:np.array)->np.array:
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def get_target(y_train:np.array) -> np.array:
    Target = np.zeros((y_train.shape[0],10))
    # line=0
    # column=0
    # for label in y_train:
    #     for column in range(0,10):
    #         if column == label:
    #             Target[line][column] = 1
    #         else:
    #             Target[line][column] = 0
    #     line += 1

    Target[np.arange(y_train.shape[0]), y_train] = 1
    return Target

Incep sa scriu componentele pentru loop-ul meu
Adica forward step, backward step si actualizari

In [3]:
def forward_step(X:np.array, w1:np.array,w2:np.array,b1:np.array,b2:np.array):
    z1 = X @ w1 + b1        # (n,100)
    y1 = ReLU(z1)           # (n,100)

    z2 = y1 @ w2 + b2       # (n,10)
    y2 = softmax(z2)        # (n,10)

    return (z1,y1,z2,y2)

In [4]:
def backward_step(X:np.array, y1:np.array, y2:np.array, T:np.array, w1:np.array, w2:np.array, lambda_reg=0.001):
    n = X.shape[0]
    eL = -(T-y2)                              # (n,10)
    gradW2 = y1.T @ eL/n + (lambda_reg * w2)  # (100,10)
    gradB2 = eL.sum(axis=0, keepdims=True)/n  # (1,10)

    e1 = deriv_ReLU(y1) * (eL @ w2.T)         # (n,100)
    gradW1 = X.T @ e1/n + (lambda_reg * w1)   # (784,100)
    gradB1 = e1.sum(axis=0, keepdims=True)/n  # (1,100)

    return (gradW1,gradB1,gradW2,gradB2)

Cross entropy pentru a masura loss ul la fiecare epoca si predicted class sa vad output-ul pt a calcula acuratetea

In [5]:
#putem sa ne folosim direct de y_train care e de forma [9,3,2,...., etc.] pentru a extrage indexul probabilitatii prezicerii corecte rapid
def cross_entropy(Y_prob:np.array, y_train:np.array)->np.array:
    m = y_train.shape[0]
    ## extrag valorile din y_prob de la 0 la m-1 la indicii din y_train
    probabilitati_corecte = Y_prob[np.arange(m), y_train]
    loss = -np.sum(np.log(probabilitati_corecte)) / m
    return loss

# Y_hat va fi matrice de (m,1) si va stabili rezultatatele prezise la fiecare imagine
def predicted_class_matrix(X:np.array, w1:np.array,w2:np.array,b1:np.array,b2:np.array) -> np.array:

    z1,y1,z2,y2 = forward_step(X,w1,w2,b1,b2)
    
    Y_hat = np.zeros((y2.shape[0],1))
    Y_hat = np.argmax(y2,axis=1,keepdims=True)
    return Y_hat



Acum incep loop-ul

In [6]:
# He Initialziaiton
w1 = np.random.randn(784, 100) * np.sqrt(2.0 / 784)  # (784,100)
b1 = np.zeros((1,100))                               # (1,100)

w2 = np.random.randn(100, 10) * np.sqrt(2.0 / 100)   # (100,10)
b2 = np.zeros((1, 10))                               # (1,10)

epoches = 120
learning_rate = 0.1
batch_size = 128
Target = get_target(y_train)

print("Starting training...")

for epoch in range(0,epoches+1):

    permutation = np.random.permutation(X_train.shape[0])

    X_shuffled = X_train[permutation]
    Y_shuffled = y_train[permutation]

    for i in range(0,X_train.shape[0],batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        Y_batch = Y_shuffled[i:i+batch_size]

        Target_batch = get_target(Y_batch)

        z1,y1,z2,y2 = forward_step(X_batch,w1,w2,b1,b2)
        gradW1, gradB1, gradW2, gradB2 = backward_step(X_batch,y1,y2,Target_batch,w1,w2)

        #update the weights
        w1 = w1 - learning_rate*gradW1
        w2 = w2 - learning_rate*gradW2
        b1 = b1 - learning_rate*gradB1
        b2 = b2 - learning_rate*gradB2

     #printuri generale
    
    if epoch % 10 == 0:
        z1,y1,z2,y2 = forward_step(X_train,w1,w2,b1,b2)
        loss = cross_entropy(y2, y_train)
        Y_hat = predicted_class_matrix(X_train,w1,w2,b1,b2)
        accuracy = np.mean(Y_hat.flatten() == y_train)
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Accuracy: {accuracy * 100:.2f}%")
    

print("Finished training")


Starting training...
Epoch 000 | Loss: 0.3041 | Accuracy: 91.50%
Epoch 010 | Loss: 0.1081 | Accuracy: 97.16%
Epoch 020 | Loss: 0.0833 | Accuracy: 97.99%
Epoch 030 | Loss: 0.0722 | Accuracy: 98.30%
Epoch 040 | Loss: 0.0675 | Accuracy: 98.50%
Epoch 050 | Loss: 0.0661 | Accuracy: 98.58%
Epoch 060 | Loss: 0.0641 | Accuracy: 98.63%
Epoch 070 | Loss: 0.0627 | Accuracy: 98.69%
Epoch 080 | Loss: 0.0648 | Accuracy: 98.58%
Epoch 090 | Loss: 0.0622 | Accuracy: 98.69%
Epoch 100 | Loss: 0.0608 | Accuracy: 98.71%
Epoch 110 | Loss: 0.0615 | Accuracy: 98.74%
Epoch 120 | Loss: 0.0598 | Accuracy: 98.75%
Finished training


Testam datele

In [7]:
import pandas as pd

Y_predicted_test = predicted_class_matrix(X_test,w1,w2,b1,b2)

submission = pd.DataFrame({
    "ID": range(len(Y_predicted_test)),
    "target": Y_predicted_test.flatten()
})

submission.to_csv("submission.csv", index=False)
print("Submission file created")


Submission file created
